# Milestone 2 — Teaching Qwen3-1.7B Hungarian with LoRA

We take the small instruction-tuned model `unsloth/Qwen3-1.7B`, test it **without training** on Hungarian
(and English) SIB-200 and Belebele, then fine-tune it with **LoRA / QLoRA** (Unsloth) and compare LoRA
settings, GPU memory and training time.

| Section | What happens |
|---|---|
| 1. Setup | imports, shared settings (all helper code is in `m2_helper.py`) |
| 2. Baselines without training | zero-shot, with the model in 4-bit and in 16-bit |
| 3. LoRA sweep | 12 configurations, one factor at a time; best one chosen on the **dev** set |
| 4. Final runs | best config trained with 3 seeds, evaluated on SIB-200 and Belebele |
| 5. Results | all tables and plots used in the README (no GPU needed) |

**How we score.** For every test example we compute the log-likelihood the model gives to each possible
answer (the 7 topic labels for SIB-200, the letters A–D for Belebele) and pick the most likely one, so the
model always gives a valid answer. For the untrained model we also let it generate freely and count how
often the output is exactly a label (`gen_valid`) and how often it is right (`gen_acc`, lenient parse).

**Metrics.** Accuracy (%) is the main metric; macro-F1 (%) for SIB-200.
*Peak GPU memory (allocated)* = `torch.cuda.max_memory_allocated()` during training: the memory the
tensors really needed. *Reserved* = `torch.cuda.max_memory_reserved()`: what PyTorch's caching allocator
held (about 2× higher with variable-length batches, and on Windows it can exceed the 6 GB of the card).
Training time = wall-clock time of `trainer.train()`, including Unsloth's one-time compilation.
Values with ± are mean ± standard deviation over seeds.

**Evaluation sets.** SIB-200: `dev` (99) and `test` (204), Hungarian and English, the same splits as in
Milestone 1. Belebele: `full` (900 questions) and `heldout` (299 questions on passages we never train on,
see `results/belebele_split.json`).

**Running it.** Every experiment runs in its own subprocess and appends one row to `results/runs.csv`
(failed runs too). Runs already in `runs.csv` are not repeated; their stored results are printed instead,
with the time they were originally run. To redo a run, delete its row.

## 1. Setup

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # Colab: get the repo + Unsloth
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "https://github.com/DeliaJing/Teaching-a-Small-Language-Model-with-LoRA.git", "/content/repo"], check=True)
    os.chdir("/content/repo/milestone2")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import m2_helper as m2
pd.set_option("display.width", 200, "display.max_columns", 40)
print("GPU:", m2.gpu_name())
split = m2.belebele_split()
print("Belebele passages: train", len(split["train_links"]), "/ held-out", len(split["heldout_links"]))

QWEN = "unsloth/Qwen3-1.7B"
PLOTS = m2.RESULTS_DIR / "plots"; PLOTS.mkdir(parents=True, exist_ok=True)

def mean_std(s):
    s = s.dropna()
    if len(s) == 0: return "–"
    return f"{s.mean():.1f}" if len(s) == 1 else f"{s.mean():.1f} ± {s.std():.1f}"

## 2. Baselines without training

The prompt is the model's own chat template, with English instructions. Labels stay in English (they are
the dataset's labels). Qwen3's "thinking" block is switched off (`enable_thinking=False`). Prompt examples:

In [ ]:
ex = m2.build_examples("sib", m2.load_sib("hun_Latn", "test").head(1), "en")[0]
print(ex["user"], "\n->", ex["answer"])
print("-" * 80)
b = m2.build_examples("belebele", m2.belebele_part("hun_Latn", "heldout").head(1), "en")[0]
print(b["user"][:900], "\n->", b["answer"])

In [ ]:
ALL_EVALS = [["sib", "hun_Latn", "dev"], ["sib", "hun_Latn", "test"], ["sib", "eng_Latn", "test"],
             ["belebele", "hun_Latn", "heldout"], ["belebele", "eng_Latn", "heldout"],
             ["belebele", "hun_Latn", "full"], ["belebele", "eng_Latn", "full"]]
GEN = ["sib_hun_test", "belebele_hun_full"]

tag, model = "qwen3-1.7b", QWEN
configs = [
    dict(run_id=f"base_{tag}_4bit_zs_en", group="baseline", kind="eval", model=model, load_in_4bit=True,
         evals=ALL_EVALS, generation=GEN, notes="zero-shot"),
]
configs.append(dict(run_id="base_qwen3-1.7b_16bit_zs_en", group="baseline", kind="eval", model=QWEN,
                    load_in_4bit=False, evals=ALL_EVALS, generation=GEN,
                    notes="zero-shot, 16-bit weights"))

for cfg in configs:
    m2.launch(cfg)

In [ ]:
runs = m2.load_runs()
cols = ["run_id", "status", "sib_hun_dev_acc", "sib_hun_test_acc", "sib_hun_test_f1", "sib_eng_test_acc",
        "belebele_hun_heldout_acc", "belebele_eng_heldout_acc", "belebele_hun_full_acc", "belebele_eng_full_acc",
        "sib_hun_test_gen_acc", "sib_hun_test_gen_valid", "belebele_hun_full_gen_acc", "belebele_hun_full_gen_valid",
        "peak_mem_load_gb", "eval_time_s"]
base = runs[runs.group == "baseline"]
base[[c for c in cols if c in base.columns]].set_index("run_id")

Reference points: XLM-R base fully fine-tuned (Milestone 1) reaches 85.6% on Hungarian SIB-200 and
89.9% on English. Random guessing is 14.3% on SIB-200 (7 labels) and 25% on Belebele (4 options).

## 3. LoRA / QLoRA sweep (SIB-200 Hungarian)

Starting point: Unsloth's free Colab notebook for Qwen3. We take its model loading
(`FastLanguageModel.from_pretrained(..., load_in_4bit=True)`), its LoRA setup (`get_peft_model`,
`lora_dropout=0`, `bias="none"`, Unsloth gradient checkpointing) and its optimizer settings (AdamW 8-bit,
linear schedule, weight decay 0.01, short warm-up). The loss is computed **only on the answer tokens**
(the idea of Unsloth's `train_on_responses_only`); we do the masking ourselves so that exactly the same
prompt is used for training and for scoring.

Default configuration: r = 16, alpha = 16, all linear layers, 4-bit, lr 2e-4, 2 epochs, effective batch 16,
seed 1. One factor is changed at a time:

| Factor | Values | Why we expect it to matter |
|---|---|---|
| rank `r` | 4, 8, **16**, 32, 64 | more capacity per layer; with only 701 examples a large rank may overfit |
| `lora_alpha` | **r**, 2r | scales the LoRA update (alpha/r); 2r = a larger effective step |
| target modules | **all linear**, attention only | the MLP layers hold much of the model's knowledge; attention-only has ~3× fewer parameters |
| precision | **4-bit (QLoRA)**, 16-bit LoRA | 4-bit should save memory at little accuracy cost |
| learning rate | 1e-4, **2e-4**, 5e-4 | Unsloth's default is 2e-4 |
| epochs | 1, **2**, 3 | small dataset: too few epochs underfit, too many overfit |

In [ ]:
DEFAULT = dict(kind="lora", model=QWEN, load_in_4bit=True, train_data="sib_hun", seed=1,
               r=16, lora_alpha=16, target="all", lr=2e-4, epochs=2, batch_size=8, grad_accum=2,
               max_seq_length=512)
FULL_EVALS = [["sib", "hun_Latn", "dev"], ["sib", "hun_Latn", "test"], ["sib", "eng_Latn", "test"],
              ["belebele", "hun_Latn", "heldout"], ["belebele", "eng_Latn", "heldout"]]

sweep = {
    "sw_default": {},
    "sw_r4": dict(r=4, lora_alpha=4), "sw_r8": dict(r=8, lora_alpha=8),
    "sw_r32": dict(r=32, lora_alpha=32), "sw_r64": dict(r=64, lora_alpha=64),
    "sw_r16_a32": dict(lora_alpha=32),
    "sw_attn": dict(target="attn"),
    "sw_16bit": dict(load_in_4bit=False),
    "sw_lr1e-4": dict(lr=1e-4), "sw_lr5e-4": dict(lr=5e-4),
    "sw_ep1": dict(epochs=1), "sw_ep3": dict(epochs=3),
}
for name, change in sweep.items():
    m2.launch({**DEFAULT, **change, "run_id": f"qwen3-1.7b_{name}", "group": "sweep",
               "notes": ", ".join(f"{k}={v}" for k, v in change.items()) or "default"})

In [ ]:
runs = m2.load_runs()
SHOW = ["run_id", "status", "r", "lora_alpha", "target_modules", "load_in_4bit", "lr", "epochs",
        "trainable_params", "trainable_pct", "sib_hun_dev_acc", "sib_hun_test_acc", "sib_hun_test_f1",
        "peak_mem_train_gb", "train_time_s", "train_loss", "error"]
sw = runs[runs.group == "sweep"]
sw[[c for c in SHOW if c in sw.columns]].set_index("run_id")

### Choosing the best configuration (dev set only; ties broken by fewer trainable parameters)

In [ ]:
ok = sw[sw.status == "ok"].sort_values(["sib_hun_dev_acc", "trainable_params"], ascending=[False, True])
best = ok.iloc[0]
BEST = {**DEFAULT, "r": int(best.r), "lora_alpha": int(best.lora_alpha), "lr": float(best.lr),
        "epochs": int(best.epochs), "target": best.target_modules, "load_in_4bit": bool(best.load_in_4bit)}
print("Best on dev:", best.run_id, f"(dev {best.sib_hun_dev_acc:.1f}%)")
print({k: BEST[k] for k in ["r", "lora_alpha", "target", "load_in_4bit", "lr", "epochs"]})

## 4. Final runs: best configuration, 3 seeds, all evaluation sets

In [ ]:
for seed in [1, 2, 3]:
    m2.launch({**BEST, "seed": seed, "run_id": f"final_qwen3-1.7b_seed{seed}", "group": "final",
               "evals": FULL_EVALS, "notes": f"best sweep config ({best.run_id})"})

## 5. Results

Tables and plots used in the README. This part only reads `results/runs.csv` and the Milestone 1 results,
so it runs in seconds without a GPU.

In [ ]:
runs = m2.load_runs()
print(len(runs), "runs,", (runs.status != "ok").sum(), "failed")

### Milestone 1 reference: XLM-R base (full fine-tuning)

In [ ]:
m1 = m2.REPO_ROOT / "milestone1" / "results"
rows = []
for lang in ["eng_Latn", "hun_Latn"]:
    ours = pd.read_csv(m1 / f"xlmr_{lang}_ours.csv") if (m1 / f"xlmr_{lang}_ours.csv").exists() else None
    authors = pd.read_csv(m1 / f"xlmr_{lang}_authors.csv") if (m1 / f"xlmr_{lang}_authors.csv").exists() else None
    rows.append({"language": lang,
                 "ours (5 seeds)": mean_std(ours.test_acc) if ours is not None else "–",
                 "authors, outputs.zip (5 seeds)": mean_std(authors.test_acc) if authors is not None else "–"})
m1_table = pd.DataFrame(rows); m1_table

### GPT-3.5 / GPT-4 reference (from the SIB-200 repository, `llm-results/`)
The authors stored the raw GPT answers. We map each answer to the first label it mentions; answers
that mention no label count as wrong. This is a rough reference (zero-shot, different prompt).

In [ ]:
def parse_gpt(text):
    t = str(text).lower()
    hits = []
    for lab in m2.SIB_LABELS:
        keys = [lab] + (["science", "technology"] if lab == "science/technology" else [])
        pos = [t.find(k) for k in keys if t.find(k) >= 0]
        if pos: hits.append((min(pos), lab))
    return min(hits)[1] if hits else None

gpt_rows = []
for model, col in [("output_gpt3.5", "gpt-3"), ("output_gpt4", "gpt-4")]:
    for lang in ["hun_Latn", "eng_Latn"]:
        d = pd.read_csv(m2.SIB_DIR / "llm-results" / model / f"{lang}.tsv", sep="\t")
        pred = d[col].map(parse_gpt)
        gpt_rows.append({"model": model.replace("output_", ""), "lang": lang,
                         "acc": round((pred == d.category).mean() * 100, 1),
                         "no label found (%)": round(pred.isna().mean() * 100, 1)})
gpt = pd.DataFrame(gpt_rows); gpt

### Main results table

In [ ]:
def summarize(df, label, method):
    return {"model": label, "method": method, "runs": len(df),
            "SIB hun test acc": mean_std(df.sib_hun_test_acc), "SIB hun test F1": mean_std(df.sib_hun_test_f1),
            "SIB eng test acc": mean_std(df.get("sib_eng_test_acc", pd.Series(dtype=float))),
            "Belebele hun held-out acc": mean_std(df.get("belebele_hun_heldout_acc", pd.Series(dtype=float))),
            "Belebele eng held-out acc": mean_std(df.get("belebele_eng_heldout_acc", pd.Series(dtype=float))),
            "trainable params": (f"{df.trainable_params.dropna().iloc[0] / 1e6:.1f}M" if df.trainable_params.notna().any() else "–"),
            "peak train mem, allocated (GB)": mean_std(df.peak_mem_alloc_train_gb) if df.peak_mem_alloc_train_gb.notna().any() else "–",
            "train time (min)": mean_std(df.train_time_s / 60) if df.train_time_s.notna().any() else "–"}

ok = runs[runs.status == "ok"]
R = lambda rid: ok[ok.run_id == rid]
P = lambda prefix: ok[ok.run_id.str.startswith(prefix)]
main_rows = []
hun_m1 = pd.read_csv(m1 / "xlmr_hun_Latn_ours.csv") if (m1 / "xlmr_hun_Latn_ours.csv").exists() else None
eng_m1 = pd.read_csv(m1 / "xlmr_eng_Latn_ours.csv")
main_rows.append({"model": "XLM-R base (278M)", "method": "full fine-tuning (M1)", "runs": 5,
                  "SIB hun test acc": mean_std(hun_m1.test_acc) if hun_m1 is not None else "–",
                  "SIB hun test F1": mean_std(hun_m1.test_f1) if hun_m1 is not None else "–",
                  "SIB eng test acc": mean_std(eng_m1.test_acc), "trainable params": "278M"})
main_rows.append(summarize(R("base_qwen3-1.7b_4bit_zs_en"), "Qwen3-1.7B", "zero-shot (4-bit)"))
main_rows.append(summarize(R("base_qwen3-1.7b_16bit_zs_en"), "Qwen3-1.7B", "zero-shot (16-bit)"))
main_rows.append(summarize(P("final_qwen3-1.7b_seed"), "Qwen3-1.7B", "QLoRA, best config, SIB hun"))
main = pd.DataFrame(main_rows).fillna("–")
main

### Sweep (Qwen3-1.7B, SIB-200 Hungarian, seed 1) — all runs, including failed ones

In [ ]:
sw = runs[runs.group == "sweep"].copy()
sweep_cols = ["run_id", "status", "notes", "trainable_params", "sib_hun_dev_acc", "sib_hun_test_acc",
              "sib_hun_test_f1", "peak_mem_alloc_train_gb", "peak_mem_train_gb", "train_time_s", "train_loss", "error"]
sweep_table = sw[[c for c in sweep_cols if c in sw.columns]].copy()
sweep_table["trainable_params"] = (sweep_table.trainable_params / 1e6).round(2)
sweep_table = sweep_table.rename(columns={"trainable_params": "trainable (M)"})
sweep_table

In [ ]:
okw = sw[sw.status == "ok"]
fig, ax = plt.subplots(figsize=(9, 4.5))
o = okw.sort_values("sib_hun_dev_acc")
ax.barh(o.run_id.str.replace("qwen3-1.7b_", ""), o.sib_hun_test_acc, color="#c9d6ea", label="test")
ax.plot(o.sib_hun_dev_acc, range(len(o)), "o", color="#2b5797", label="dev")
zs = R("base_qwen3-1.7b_4bit_zs_en")
if len(zs): ax.axvline(zs.sib_hun_test_acc.iloc[0], color="gray", ls="--", label="zero-shot (test)")
if hun_m1 is not None: ax.axvline(hun_m1.test_acc.mean(), color="#b03a2e", ls=":", label="XLM-R base FT (test)")
ax.set_xlabel("SIB-200 hun accuracy (%)"); ax.set_title("All sweep configurations (Qwen3-1.7B, seed 1)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, frameon=False); plt.tight_layout(); plt.savefig(PLOTS / "sweep_all.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
mem = okw[["run_id", "peak_mem_alloc_train_gb", "train_time_s", "trainable_params", "sib_hun_test_acc"]].copy()
mem["run"] = mem.run_id.str.replace("qwen3-1.7b_", "")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(mem.peak_mem_alloc_train_gb, mem.sib_hun_test_acc)
for _, r in mem.iterrows(): ax.annotate(r.run, (r.peak_mem_alloc_train_gb, r.sib_hun_test_acc), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("peak GPU memory allocated during training (GB)"); ax.set_ylabel("SIB-200 hun test accuracy (%)")
ax.set_title("Memory vs accuracy"); plt.tight_layout(); plt.savefig(PLOTS / "memory_vs_accuracy.png", dpi=130); plt.show()

### Format following of the untrained model (free generation)

In [ ]:
g = ok[ok.group == "baseline"][["run_id", "sib_hun_test_acc", "sib_hun_test_gen_acc", "sib_hun_test_gen_valid",
                                "belebele_hun_full_acc", "belebele_hun_full_gen_acc", "belebele_hun_full_gen_valid"]]
g = g.dropna(subset=["sib_hun_test_gen_acc"]); g

### Confusion matrix: best Qwen3 LoRA (seed 1) vs zero-shot, SIB-200 hun test

In [ ]:
from sklearn.metrics import confusion_matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, rid, title in [(axes[0], "base_qwen3-1.7b_4bit_zs_en", "zero-shot"), (axes[1], "final_qwen3-1.7b_seed1", "QLoRA (best, seed 1)")]:
    f = m2.PRED_DIR / f"{rid}__sib_hun_Latn_test.csv"
    if not f.exists(): continue
    p = pd.read_csv(f)
    cm = confusion_matrix(p.gold, p.pred, labels=m2.SIB_LABELS)
    ax.imshow(cm, cmap="Blues")
    for i in range(7):
        for j in range(7): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "black")
    short = [l.split("/")[0][:8] for l in m2.SIB_LABELS]
    ax.set_xticks(range(7), short, rotation=45); ax.set_yticks(range(7), short)
    ax.set_xlabel("predicted"); ax.set_ylabel("gold"); ax.set_title(f"{title}: acc {(p.gold == p.pred).mean() * 100:.1f}%")
plt.tight_layout(); plt.savefig(PLOTS / "confusion_qwen.png", dpi=130); plt.show()

### Export tables for the README

In [ ]:
with open(m2.RESULTS_DIR / "tables.md", "w", encoding="utf-8") as f:
    for title, t in [("Milestone 1", m1_table), ("GPT reference", gpt), ("Main results", main),
                     ("Sweep", sweep_table), ("Format following", g)]:
        f.write(f"## {title}\n\n{t.to_markdown(index=False)}\n\n")
print(open(m2.RESULTS_DIR / "tables.md", encoding="utf-8").read())